In [1]:
using MarineHydro
using PyCall
# import your capytaine mesh
cpt = pyimport("capytaine")
radius = 1.0 #fixed
resolution = (10, 10)
cptmesh = cpt.mesh_sphere(name="sphere", radius=radius, center=(0, 0, 0), resolution=resolution) 
cptmesh.keep_immersed_part(inplace=true)

# declare it Julia mesh
mesh = Mesh(cptmesh)  
ω = 1.03
ζ = [0,0,1] # HEAVE: will be more verbose in future iteration. define it again even if defined in Capytaine.
F = DiffractionForce(mesh,ω,ζ)
A,B = calculate_radiation_forces(mesh,ζ,ω)

2-element Vector{Float64}:
 1663.1322676227107
  374.48616666382367

In [2]:
function hydro_coeffs(mesh,ω,ζ)
    F = DiffractionForce(mesh,ω,ζ)
    A,B = calculate_radiation_forces(mesh,ζ,ω)
    return A, B, F
end


hydro_coeffs (generic function with 1 method)

In [3]:
A, B, F = hydro_coeffs(mesh,ω,ζ)

(1663.1322676227107, 374.48616666382367, -1699.6336651706672 - 377.86919323663113im)

In [4]:
# import Pkg
# Pkg.add("Zygote")
# using Zygote
# A_w_grad, = Zygote.gradient(w -> calculate_radiation_forces(mesh,ζ,w)[1],ω)

In [5]:
# Method 1: Not In Parallel

# Create range object (start_omega:incriment:end_omega)
frequencies = range(0.1, 10.0, length=20)

# Method 1.1: Broadcast over frequency range. The . after the function indicates it will
# run the function once for each value in the list. Ref() indicates that mesh
# and ζ are constants, while frequencies is looped over.
results = hydro_coeffs.(Ref(mesh), frequencies, Ref(ζ))


# Method 1.2: Typical loop structure. Call hydro_coeffs for current frequency, 
# takes output of function and appends to end of results list. The 
# ! indicates that results changes.
# results = []
# for ω in frequencies
#     push!(results, hydro_coeffs(mesh, ω, ζ))
# end


20-element Vector{Tuple{Float64, Float64, ComplexF64}}:
 (1614.1586247292128, 0.4496829866230369, -15.776336596364306 - 0.0439572902330041im)
 (1690.651747805288, 97.96549106229128, -634.0164767864298 - 59.49163052141615im)
 (1635.733160482102, 478.92752836097935, -2047.9983696861723 - 536.4213879590762im)
 (1424.3505554079998, 1017.599826343355, -3683.007270139725 - 1676.3950210005632im)
 (1166.107220788045, 1436.0937769395282, -4910.103401607666 - 3172.174292483789im)
 (955.4255544865435, 1590.8808476458853, -5482.36798711197 - 4488.634591148132im)
 (823.7653004306162, 1507.2020601386853, -5452.542429684718 - 5238.232224936268im)
 (762.0190111536372, 1284.9497420607875, -4980.757229436386 - 5269.897243724504im)
 (746.2432597404321, 1015.1247283009842, -4222.675528236203 - 4614.112648229189im)
 (750.8559567005063, 740.9429250711507, -3300.384113317298 - 3441.1748113832346im)
 (804.58483393605, 708.0681915529211, -2301.754501232997 - 2179.900165067008im)
 (810.5240869775291, 459.468456

In [ ]:
# Method 2: In parallel using CPU (multithreading). Memory is 
# shared between threads. This 
# distrubutes computations to different CPU cores so that 
# the function can be evaluated at multiple frequencies 
# simultaneously. 

# ENV["JULIA_NUM_THREADS"] = "auto"  # Or "auto" to use all your cores

# make @threads available
using Base.Threads

# get number of frequencies 
n = length(frequencies)

# Create an array that can hold any type of data. 
# (undef, n) says to reserve n spots.
results = Vector{Any}(undef, n)

# @threads desides how many threads are available
# and distributes the frequencies to each thread.
@threads for i in 1:n
    # threadid() says which thread (frequency evaluation). 
    # nthreads() says the total number of threads that can be worked on at once.
    println("Working on index $i on thread $(threadid()) of $(nthreads())")
    # Each thread handles a different index 'i'
    results[i] = hydro_coeffs(mesh, frequencies[i], ζ)
end

# Note: I had to follow these steps to get this to use more than one core:
# Open Settings (Ctrl + ,)
# Search for julia.numThreads
# Change it from 1 to "auto"

# Note that this is noticably faster than the previous 
# cell that does sequential evaluations of the function.


Working on index 20 on thread 1 of 16
Working on index 7 on thread 12 of 16
Working on index 19 on thread 7 of 16
Working on index 5 on thread 11 of 16
Working on index 15 on thread 10 of 16
Working on index 9 on thread 3 of 16
Working on index 18 on thread 16 of 16
Working on index 1 on thread 2 of 16
Working on index 11 on thread 9 of 16
Working on index 10 on thread 6 of 16
Working on index 13 on thread 8 of 16
Working on index 12 on thread 5 of 16
Working on index 16 on thread 13 of 16
Working on index 17 on thread 14 of 16
Working on index 3 on thread 15 of 16
Working on index 14 on thread 4 of 16
Working on index 2 on thread 2 of 16
Working on index 8 on thread 12 of 16
Working on index 6 on thread 11 of 16
Working on index 4 on thread 15 of 16


In [14]:
using CUDA
CUDA.versioninfo()
# d_frequencies = CuArray(frequencies)

ErrorException: CUDA driver not found

In [ ]:
# Method 3: In parallel using GPU. GPU has its own version of RAM (VRAM). 
# For each frequency, each thread will need to be at the same stage 
# at the same time. First, the data is moved from RAM to VRAM. Then 
# computations happen in VRAM. Then the outputs are moved back to RAM.

using CUDA

# Move your data from RAM to VRAM.
d_frequencies = CuArray(frequencies)
d_mesh = cu(mesh) 
d_ζ = cu(ζ)

# Run the function in VRAM
# d_results = hydro_coeffs.(Ref(d_mesh), d_frequencies, Ref(d_ζ))
results = cos(d_frequencies)

# MOve data from VRAM to RAM
results = Array(d_results)

ErrorException: CUDA driver not found